In [ ]:
# =========================================================
# Windows EXE 빌드 패키지 생성
#
# Colab 실행 결과:
#   FNN_EXE_Package.zip 다운로드
#
# Windows에서 ZIP 압축 해제 후:
#   build_exe.bat 실행
#
# 최종 생성 파일:
#   exe_folder/fnn_model.exe
# =========================================================
from pathlib import Path
import shutil, os
from google.colab import files


# GitHub repository를 저장할 위치
REPO_ROOT = Path("/content/Basic_example_copy").resolve()

# 저장소가 없으면 clone
if not os.path.exists(REPO_ROOT):
    !git clone -q https://github.com/Jaehoon-Shim/Basic_example.git {REPO_ROOT}

BASE_DIR = REPO_ROOT
RESULT_DIR = BASE_DIR / "results"
PT_PATH = RESULT_DIR/"fnn_model.pt"
PACKAGE_DIR = RESULT_DIR / "FNN_EXE_Package"
ZIP_BASE = RESULT_DIR / "FNN_EXE_Package"

if not PT_PATH.exists():
    raise FileNotFoundError(
        f"모델 파일을 찾을 수 없습니다.\n"
        f"확인 경로: {PT_PATH}"
    )


# =========================================================
# Windows에서 실행될 실제 GUI 프로그램
# =========================================================
APP_CODE = r'''import sys
from pathlib import Path

import torch
import torch.nn as nn
from PyQt6.QtCore import Qt
from PyQt6.QtWidgets import (
    QApplication,
    QLabel,
    QLineEdit,
    QMessageBox,
    QPushButton,
    QVBoxLayout,
    QWidget,
)


# =========================================================
# StandardScaler parameters
# =========================================================
X_MEAN = 20.0
X_SCALE = 11.5759087

Y_MEAN = 5.44184351
Y_SCALE = 3.66043448


# =========================================================
# Resource path
# =========================================================
def resource_path(filename):
    """
    일반 Python, Notebook 및 PyInstaller 실행환경의
    리소스 경로를 처리합니다.
    """

    # PyInstaller onefile 또는 onedir 실행
    if getattr(sys, "frozen", False):
        base_path = Path(sys._MEIPASS)

    # 일반 .py 파일 실행
    elif "__file__" in globals():
        base_path = Path(__file__).resolve().parent

    # Notebook 환경
    else:
        base_path = Path.cwd()

    return base_path / filename


# =========================================================
# FNN Model
# Architecture: 1 -> 8 -> 8 -> 8 -> 1
# =========================================================
class FNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),

            nn.Linear(8, 8),
            nn.ReLU(),

            nn.Linear(8, 8),
            nn.ReLU(),

            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.model(x)


# =========================================================
# GUI Application
# =========================================================
class FNNPredictorApp(QWidget):
    def __init__(self):
        super().__init__()

        self.model = FNN()
        self.load_model()
        self.initialize_ui()

    def load_model(self):
        model_path = resource_path("fnn_model.pt")

        if not model_path.exists():
            raise FileNotFoundError(
                f"fnn_model.pt를 찾을 수 없습니다.\n\n"
                f"확인 경로:\n{model_path}"
            )

        checkpoint = torch.load(
            model_path,
            map_location="cpu",
            weights_only=True,
        )

        # 다음 두 저장 형식을 모두 지원합니다.
        #
        # 1. torch.save(model.state_dict(), path)
        # 2. torch.save(
        #        {"model_state_dict": model.state_dict()},
        #        path
        #    )
        if (
            isinstance(checkpoint, dict)
            and "model_state_dict" in checkpoint
        ):
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint

        self.model.load_state_dict(state_dict)
        self.model.to("cpu")
        self.model.eval()

    def initialize_ui(self):
        self.setWindowTitle("FNN Model Inference")
        self.setFixedSize(420, 400)

        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(35, 30, 35, 25)
        main_layout.setSpacing(16)

        title_label = QLabel("추론 프로그램")
        title_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        title_label.setStyleSheet(
            "font-size: 22px;"
            "font-weight: bold;"
            "color: #0B2A53;"
            "padding: 8px;"
        )
        main_layout.addWidget(title_label)

        input_label = QLabel("신경망 입력값")
        input_label.setStyleSheet(
            "font-size: 15px;"
            "font-weight: bold;"
        )
        main_layout.addWidget(input_label)

        self.input_edit = QLineEdit()
        self.input_edit.setPlaceholderText(
            "숫자를 입력하세요."
        )
        self.input_edit.setStyleSheet(
            "font-size: 16px;"
            "padding: 8px;"
        )
        self.input_edit.returnPressed.connect(
            self.run_inference
        )
        main_layout.addWidget(self.input_edit)

        predict_button = QPushButton("예측 실행")
        predict_button.setStyleSheet(
            "QPushButton {"
            "font-size: 16px;"
            "font-weight: bold;"
            "color: white;"
            "background-color: #0B2A53;"
            "border-radius: 5px;"
            "padding: 10px;"
            "}"
            "QPushButton:hover {"
            "background-color: #174D89;"
            "}"
        )
        predict_button.clicked.connect(
            self.run_inference
        )
        main_layout.addWidget(predict_button)

        self.result_label = QLabel("예측 결과: -")
        self.result_label.setAlignment(
            Qt.AlignmentFlag.AlignCenter
        )
        self.result_label.setStyleSheet(
            "font-size: 18px;"
            "font-weight: bold;"
            "color: #174D89;"
            "border: 1px solid #AAB7C4;"
            "border-radius: 5px;"
            "padding: 15px;"
        )
        main_layout.addWidget(self.result_label)

        developer_label = QLabel(
            "Developed by Jaehoon Shim (SEED Lab)"
        )
        developer_label.setAlignment(
            Qt.AlignmentFlag.AlignRight
        )
        developer_label.setStyleSheet(
            "font-size: 11px;"
            "color: #888888;"
            "padding: 2px 4px;"
        )
        main_layout.addWidget(developer_label)

        self.setLayout(main_layout)
        self.input_edit.setFocus()

    def run_inference(self):
        try:
            x_value = float(
                self.input_edit.text().strip()
            )

            # 입력 표준화
            x_scaled = (
                x_value - X_MEAN
            ) / X_SCALE

            x_tensor = torch.tensor(
                [[x_scaled]],
                dtype=torch.float32,
                device="cpu",
            )

            # 추론
            with torch.inference_mode():
                y_scaled = self.model(
                    x_tensor
                ).item()

            # 출력 역표준화
            y_prediction = (
                y_scaled * Y_SCALE
                + Y_MEAN
            )

            self.result_label.setText(
                f"예측 결과: {y_prediction:.8g}"
            )

        except ValueError:
            QMessageBox.warning(
                self,
                "입력 오류",
                "숫자를 입력해 주세요.",
            )

        except Exception as error:
            QMessageBox.critical(
                self,
                "추론 오류",
                str(error),
            )


def main():
    app = QApplication(sys.argv)

    try:
        window = FNNPredictorApp()
        window.show()
        sys.exit(app.exec())

    except Exception as error:
        QMessageBox.critical(
            None,
            "프로그램 실행 오류",
            str(error),
        )
        sys.exit(1)


if __name__ == "__main__":
    main()
'''


# =========================================================
# Windows EXE 생성용 BAT 파일
# =========================================================
BUILD_BAT = r'''@echo off
setlocal
chcp 65001 > nul
cd /d "%~dp0"

echo ========================================
echo FNN Windows EXE Build
echo ========================================
echo.

if not exist "fnn_model.pt" (
    echo [오류] fnn_model.pt 파일이 없습니다.
    pause
    exit /b 1
)

if not exist "FNN_Inference.py" (
    echo [오류] FNN_Inference.py 파일이 없습니다.
    pause
    exit /b 1
)

REM ----------------------------------------
REM 출력 폴더를 빌드 전에 생성
REM ----------------------------------------
if not exist "exe_folder" (
    mkdir "exe_folder"
)

REM ----------------------------------------
REM Python 실행 명령 확인
REM ----------------------------------------
set "PYTHON_CMD="

where py >nul 2>nul
if not errorlevel 1 (
    set "PYTHON_CMD=py -3"
)

if not defined PYTHON_CMD (
    where python >nul 2>nul
    if not errorlevel 1 (
        set "PYTHON_CMD=python"
    )
)

REM ----------------------------------------
REM Python이 없으면 자동 설치
REM ----------------------------------------
if not defined PYTHON_CMD (
    echo Python이 없습니다.
    echo Python 3.12 자동 설치를 시작합니다.
    echo.

    where winget >nul 2>nul

    if errorlevel 1 (
        echo [오류] winget을 사용할 수 없습니다.
        echo Microsoft Store 또는 python.org에서
        echo Python 3.12를 설치한 후 다시 실행하세요.
        pause
        exit /b 1
    )

    winget install ^
      --id Python.Python.3.12 ^
      -e ^
      --scope user ^
      --silent ^
      --accept-package-agreements ^
      --accept-source-agreements

    if errorlevel 1 (
        echo [오류] Python 자동 설치에 실패했습니다.
        pause
        exit /b 1
    )

    echo.
    echo Python 설치가 완료되었습니다.
    echo 현재 창에서 Python이 인식되지 않으면
    echo 창을 닫고 build_exe.bat을 다시 실행하세요.
    echo.

    set "PYTHON_CMD=py -3"
)

REM ----------------------------------------
REM Python 실행 확인
REM ----------------------------------------
echo Python 환경을 확인합니다.
%PYTHON_CMD% --version

if errorlevel 1 (
    echo.
    echo [오류] Python을 실행할 수 없습니다.
    echo 이 창을 닫은 후 build_exe.bat을
    echo 다시 실행해 주세요.
    pause
    exit /b 1
)

REM ----------------------------------------
REM pip 준비
REM ----------------------------------------
echo.
echo pip를 준비합니다.

%PYTHON_CMD% -m ensurepip --upgrade
%PYTHON_CMD% -m pip install --upgrade pip

if errorlevel 1 (
    echo [오류] pip 준비에 실패했습니다.
    pause
    exit /b 1
)

REM ----------------------------------------
REM CPU 전용 PyTorch 설치
REM ----------------------------------------
echo.
echo PyTorch 설치 여부를 확인합니다.

%PYTHON_CMD% -c "import torch" >nul 2>nul

if errorlevel 1 (
    echo CPU 전용 PyTorch를 설치합니다.

    %PYTHON_CMD% -m pip install ^
      torch ^
      --index-url https://download.pytorch.org/whl/cpu

    if errorlevel 1 (
        echo [오류] PyTorch 설치에 실패했습니다.
        pause
        exit /b 1
    )
)

REM ----------------------------------------
REM PyQt6 설치
REM ----------------------------------------
echo.
echo PyQt6 설치 여부를 확인합니다.

%PYTHON_CMD% -c "import PyQt6" >nul 2>nul

if errorlevel 1 (
    echo PyQt6를 설치합니다.
    %PYTHON_CMD% -m pip install PyQt6

    if errorlevel 1 (
        echo [오류] PyQt6 설치에 실패했습니다.
        pause
        exit /b 1
    )
)

REM ----------------------------------------
REM PyInstaller 설치
REM ----------------------------------------
echo.
echo PyInstaller 설치 여부를 확인합니다.

%PYTHON_CMD% -c "import PyInstaller" >nul 2>nul

if errorlevel 1 (
    echo PyInstaller를 설치합니다.
    %PYTHON_CMD% -m pip install pyinstaller

    if errorlevel 1 (
        echo [오류] PyInstaller 설치에 실패했습니다.
        pause
        exit /b 1
    )
)

REM ----------------------------------------
REM Windows EXE 생성
REM ----------------------------------------
echo.
echo fnn_model.exe 생성을 시작합니다.
echo PyTorch 포함으로 시간이 걸릴 수 있습니다.
echo.

%PYTHON_CMD% -m PyInstaller ^
  --noconfirm ^
  --clean ^
  --onefile ^
  --windowed ^
  --name fnn_model ^
  --distpath "exe_folder" ^
  --add-data "fnn_model.pt;." ^
  --collect-all torch ^
  FNN_Inference.py

if errorlevel 1 (
    echo.
    echo [오류] EXE 생성에 실패했습니다.
    pause
    exit /b 1
)

echo.
echo ========================================
echo EXE 생성 완료
echo.
echo 결과:
echo exe_folder\fnn_model.exe
echo.
echo fnn_model.pt는 EXE 내부에 포함되었습니다.
echo 최종 실행 PC에는 Python과 PyTorch가 필요하지 않습니다.
echo ========================================
pause
'''


# =========================================================
# 패키지 폴더 생성
# =========================================================

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# 모델 복사
shutil.copy2(
    PT_PATH,
    PACKAGE_DIR / "fnn_model.pt",
)

# 실행 프로그램 저장
with open(
    PACKAGE_DIR / "FNN_Inference.py",
    "w",
    encoding="utf-8",
) as file:
    file.write(APP_CODE)

# BAT 파일 저장
with open(
    PACKAGE_DIR / "build_exe.bat",
    "w",
    encoding="utf-8",
    newline="\r\n",
) as file:
    file.write(BUILD_BAT)


# =========================================================
# ZIP 파일 생성
# =========================================================
ZIP_PATH = Path(
    shutil.make_archive(
        str(ZIP_BASE),
        "zip",
        PACKAGE_DIR,
    )
)

print("모델:", PT_PATH)
print("Windows 빌드 패키지:", ZIP_PATH)
print()
print("사용 방법:")
print("1. ZIP 파일의 압축을 Windows PC에서 해제")
print("2. build_exe.bat 실행")
print("3. exe_folder/fnn_model.exe 확인")

files.download(str(ZIP_PATH))

ModuleNotFoundError: No module named 'google'